# Automated Food Desert (LILA) Analysis — Any State

**This one notebook replaces** `Calculation_Metro_Income_Indiana.ipynb`, `Designation_Metro_Indiana.ipynb`,
and `Grided_LILA_Analysis_Comparison.ipynb`. **Nothing is downloaded manually** — every dataset is read
straight from a URL / the Census API with `pandas` / `geopandas`.

### How to use (when you come back in a week)
1. In **Section 1**, set `STATE_FIPS` and `STATE_NAME` (that's it — everything else adapts).
2. Run all cells top to bottom. Sections 6–7 (grid + Overpass) are the slow ones (see runtime notes in each section).
3. Section 10 makes the Low-Income / Low-Access / LILA maps; Section 11 checks accuracy against the official USDA 2019 Food Access Research Atlas (read directly from the ERS URL).

### Pipeline
| Step | What | Source (auto-fetched) |
|---|---|---|
| 2 | Tract boundaries | TIGER 2019 zip URL |
| 3 | Poverty %, median family income, population, households, no-vehicle households | Census API, 2019 ACS-5 profile |
| 3 | **State** & **Metro (CBSA)** median family income | Census API (state / CBSA geography — no more merging 6 states' shapefiles!) |
| 3 | Urban / rural status | Census API, 2010 Decennial SF1 (P002) |
| 4 | Metro tract designation | TIGER 2019 CBSA shapefile (centroid-in-metro test — replaces the manual GIS pairwise-intersect step) |
| 6 | ½-km population grid | built in-notebook (EPSG:5070) |
| 7 | Supermarket locations | OpenStreetMap Overpass API (state bbox **+ 20-mile buffer**; pinned to a historical snapshot date via Overpass **attic data** so stores match the analysis year) |
| 8 | Distance to nearest store | straight-line grid-to-store (SciPy KDTree) — **this is what USDA does**; they do *not* use road networks |
| 11 | Ground truth for accuracy | USDA FARA 2019 `.xlsx` read directly with pandas |

### USDA definitions implemented (2019 Atlas)
- **Low-income tract** (Treasury NMTC): poverty rate ≥ 20 %, **or** median family income ≤ 80 % of the *state* median family income, **or** (metro tract) ≤ 80 % of the *metro-area* median family income.
- **Low-access (1-and-10)** — computed for **every** tract: ≥ 500 people **or** ≥ 33 % of population farther than **1 mile** (urban tract) / **10 miles** (rural tract) from the nearest supermarket.
- **Vehicle measure**: ≥ 100 households > ½ mile from a store with no vehicle, **or** ≥ 500 people / 33 % beyond 20 miles.
- **LILA** = low income **and** low access.
- **Urban tract** = tract centroid in a Census urban area (≥ 2,500 people). We proxy this with the tract's 2010 urban-population share (> 50 %) from the SF1 API — see note in Section 3.

Docs: https://www.ers.usda.gov/data-products/food-access-research-atlas/documentation

**Requirements:** `pip install geopandas matplotlib scipy requests openpyxl` — `osmnx`/`networkx` are **no longer needed**.

---
# 1 — Setup & Parameters

Change `STATE_FIPS` / `STATE_NAME` to run any state. FIPS codes: https://transition.fcc.gov/oet/info/maps/census/fips/fips.txt

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap
import shapely
from shapely.geometry import Point
from scipy.spatial import cKDTree
import requests

In [ ]:
# ============================== CHANGE THESE ==============================
STATE_FIPS = "18"          # 18 = Indiana
STATE_NAME = "Indiana"     # must match the "State" column in the USDA file (full name)

API_KEY    = "d611946a01ad74390c6fc340807a5377826f8a07"   # your free Census key

# ====================== usually leave everything below ====================
YEAR            = "2019"      # latest year the USDA Food Access Research Atlas covers
ACS_BASE        = f"https://api.census.gov/data/{YEAR}/acs/acs5/profile"
EQUAL_AREA_CRS  = "EPSG:5070" # Albers Equal Area (meters) — for grids & distances
CELL_SIZE       = 500         # ½-km grid, same as USDA
M_PER_MILE      = 1609.344

# OSM store types counted as "supermarket / supercenter / large grocery".
# shop=supermarket covers Kroger/Meijer/Walmart Supercenter etc.; wholesale = Costco/Sam's.
# (The old notebook also queried retail/farm/health_food/department_store — far too broad.)
SHOP_TAGS          = ["supermarket", "wholesale"]
STORE_BUFFER_MILES = 20       # also grab stores this far OUTSIDE the state (border tracts!)

# Historical OSM snapshot ("attic data"): fetch the supermarkets that existed in OSM ON THIS
# DATE, so store data matches the analysis year. Defaults to the end of YEAR (end-2019 for the
# USDA comparison). Set to None for the present-day map (e.g. a "2019 vs today" change study).
STORE_SNAPSHOT_DATE = f"{YEAR}-12-31T23:59:59Z"   # or None for current data

# USDA distance thresholds (miles)
URBAN_DIST, RURAL_DIST, HALF_MILE, FAR_DIST = 1.0, 10.0, 0.5, 20.0

### Map helper

Adds the standard map elements (lat/lon labels, north arrow, scale bar) to any axis — reused on
every map below. `ylabel=False` suppresses the "Latitude" label on all but the first panel of
multi-panel figures.

In [ ]:
def add_map_elements(ax, bar_km=50, ylabel=True):
    """Lat/lon labels, north arrow, and scale bar (same helper as the test notebook)."""
    ax.set_xlabel("Longitude")
    if ylabel:
        ax.set_ylabel("Latitude")
    ax.annotate("", xy=(0.95, 0.93), xytext=(0.95, 0.85), xycoords="axes fraction",
                arrowprops=dict(arrowstyle="-|>", color="black", lw=1.5))
    ax.text(0.95, 0.94, "N", transform=ax.transAxes, ha="center",
            fontsize=11, fontweight="bold")
    xmin, xmax = ax.get_xlim(); ymin, ymax = ax.get_ylim()
    km_per_deg = 111.32 * np.cos(np.radians((ymin + ymax) / 2))
    bar_deg = bar_km / km_per_deg
    x0 = xmin + (xmax - xmin) * 0.05; y0 = ymin + (ymax - ymin) * 0.05
    ax.plot([x0, x0 + bar_deg], [y0, y0], color="black", lw=3)
    ax.text(x0 + bar_deg / 2, y0 + (ymax - ymin) * 0.015, f"{bar_km} km",
            ha="center", fontsize=8)

---
# 2 — Tract Boundaries (TIGER 2019, read straight from the zip URL)

In [ ]:
tracts = gpd.read_file(
    f"https://www2.census.gov/geo/tiger/TIGER{YEAR}/TRACT/tl_{YEAR}_{STATE_FIPS}_tract.zip"
)
tracts["GEOID"] = tracts["GEOID"].astype(str)
print(f"{len(tracts)} tracts loaded for {STATE_NAME}")
tracts.head(3)

---
# 3 — All Census Data via the API (no manual downloads)

### Variables actually needed (from your list)
| Code | Name here | Used for |
|---|---|---|
| `DP05_0001E` | `totalpop` | tract population (grid allocation, 500-people/33 % tests) |
| `DP03_0128PE` | `percentpov` | poverty-rate ≥ 20 % test (the `PE` version gives the percent directly) |
| `DP03_0086E` | `medfam` | median family income tests |
| `DP02_0001E` | `tothouseholds` | household allocation to grid |
| `DP04_0058E` | `novehicle` | no-vehicle-household measure |

**Dropped as unnecessary:** `DP03_0062E/0063E/0087E` (household/mean incomes — NMTC uses *median family* only),
`DP03_0051E`, `DP02_0002E/0018E`, `DP04_0001E/0002E/xxPE`, `DP04_0057E`, `DP03_0128E` (redundant with the `PE` version).

In [ ]:
def get_census(url):
    """Download a Census API table into a clean DataFrame (first row = header)."""
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    data = r.json()
    return pd.DataFrame(data[1:], columns=data[0])

TRACT_VARS = {
    "DP05_0001E":  "totalpop",
    "DP03_0128PE": "percentpov",
    "DP03_0086E":  "medfam",
    "DP02_0001E":  "tothouseholds",
    "DP04_0058E":  "novehicle",
}

url = (f"{ACS_BASE}?get=NAME,GEO_ID,{','.join(TRACT_VARS)}"
       f"&for=tract:*&in=state:{STATE_FIPS}%20county:*&key={API_KEY}")
acs = get_census(url).rename(columns=TRACT_VARS)
acs["GEOID"] = acs["GEO_ID"].str.split("US").str[1]

# numbers arrive as text; Census sentinels like -666666666 mean "no estimate" -> NaN
for c in TRACT_VARS.values():
    acs[c] = pd.to_numeric(acs[c], errors="coerce")
    acs[c] = acs[c].mask(acs[c] < 0)

print(f"{len(acs)} tract rows; missing medfam in {acs['medfam'].isna().sum()} tracts")
acs.head(3)

### 3.2 — State & Metro-area median family income (one API call each)

The old workflow approximated these with the *median of tract medians* (and needed shapefiles of all
6 neighboring states + manual GIS intersection to do it). The Census publishes the actual ACS value
directly at the **state** and **CBSA** geography levels — so we just ask for it.

In [ ]:
# State median family income
state_row = get_census(f"{ACS_BASE}?get=DP03_0086E&for=state:{STATE_FIPS}&key={API_KEY}")
STATE_MEDFAM = float(state_row["DP03_0086E"].iloc[0])
print(f"{STATE_NAME} median family income ({YEAR} ACS-5): ${STATE_MEDFAM:,.0f}")

# Median family income for every metro/micro area in the US (we filter to metros when joining)
cbsa_geo = "metropolitan%20statistical%20area/micropolitan%20statistical%20area"
cbsa_inc = get_census(f"{ACS_BASE}?get=NAME,DP03_0086E&for={cbsa_geo}:*&key={API_KEY}")
cbsa_inc.columns = ["CBSA_NAME", "cbsa_medfam", "CBSAFP"]
cbsa_inc["cbsa_medfam"] = pd.to_numeric(cbsa_inc["cbsa_medfam"], errors="coerce").mask(lambda s: s < 0)
print(f"{len(cbsa_inc)} CBSAs downloaded")
cbsa_inc.head(3)

### 3.3 — Urban / rural status (2010 Decennial SF1)

USDA: a tract is **urban** if its centroid falls inside a Census urban area (≥ 2,500 people); everything
else is rural. *(The old notebook used `totalpop < 2500` — that's the tract's own population, which is a
different thing entirely and misclassifies most rural tracts, since a typical tract holds ~4,000 people.)*

Proxy used here: SF1 table `P002` gives each tract's population living inside urban areas — we call a
tract urban when **> 50 %** of its 2010 population was urban. This agrees with the centroid test for the
vast majority of tracts and needs only a tiny API call instead of the ~200 MB national urban-areas shapefile
(`tl_2019_us_uac10.zip` — swap it in via a centroid `sjoin` if you ever want the exact test).

In [ ]:
sf1 = get_census(
    f"https://api.census.gov/data/2010/dec/sf1?get=P002001,P002002"
    f"&for=tract:*&in=state:{STATE_FIPS}%20county:*&key={API_KEY}"
)
sf1["GEOID"] = sf1["state"] + sf1["county"] + sf1["tract"]
for c in ("P002001", "P002002"):
    sf1[c] = pd.to_numeric(sf1[c], errors="coerce")
sf1["urban"] = (sf1["P002002"] / sf1["P002001"].replace(0, np.nan) > 0.5).fillna(False)

acs = acs.merge(sf1[["GEOID", "urban"]], on="GEOID", how="left")
acs["urban"] = acs["urban"].fillna(False)   # unmatched (rare boundary changes) -> rural
print(acs["urban"].value_counts())

---
# 4 — Metro Designation (replaces both prerequisite notebooks)

A tract is "in a metropolitan area" if it lies inside a **Metropolitan** Statistical Area
(CBSA shapefile, `LSAD == 'M1'`; `'M2'` = micropolitan, which does **not** count).
We test each tract's representative point against the metro polygons, then attach that
metro's ACS median family income from Section 3.2.

In [ ]:
cbsa = gpd.read_file(f"https://www2.census.gov/geo/tiger/TIGER{YEAR}/CBSA/tl_{YEAR}_us_cbsa.zip")
metros = cbsa[cbsa["LSAD"] == "M1"][["CBSAFP", "NAME", "geometry"]].rename(columns={"NAME": "metro_name"})

pts = tracts[["GEOID"]].copy()
pts["geometry"] = tracts.geometry.representative_point()   # always inside the tract
pts = gpd.GeoDataFrame(pts, crs=tracts.crs).to_crs(metros.crs)

metro_join = gpd.sjoin(pts, metros, how="left", predicate="within")
metro_join = metro_join.drop_duplicates("GEOID")[["GEOID", "CBSAFP", "metro_name"]]
metro_join = metro_join.merge(cbsa_inc[["CBSAFP", "cbsa_medfam"]], on="CBSAFP", how="left")

acs = acs.merge(metro_join, on="GEOID", how="left")
print(f"Metro tracts: {acs['CBSAFP'].notna().sum()} / {len(acs)}")
acs["metro_name"].value_counts()

---
# 5 — Low-Income Classification (NMTC rule)

In [ ]:
acs["low_income"] = (
    (acs["percentpov"] >= 20)
    | (acs["medfam"] <= 0.8 * STATE_MEDFAM)
    | (acs["CBSAFP"].notna() & (acs["medfam"] <= 0.8 * acs["cbsa_medfam"]))
).astype(int)

tracts = tracts.merge(
    acs[["GEOID", "totalpop", "percentpov", "medfam", "tothouseholds",
         "novehicle", "urban", "CBSAFP", "metro_name", "low_income"]],
    on="GEOID", how="left"
)
print(f"Low-income tracts: {tracts['low_income'].sum()} / {len(tracts)}")

---
# 6 — ½-km Grid & Population Allocation

Build a 500 m grid over the state (in equal-area EPSG:5070, origin snapped to 500 m so it's
reproducible), intersect with tracts, and allocate each tract's population / households /
no-vehicle households to the pieces **proportionally to area** (uniform-density assumption;
USDA allocates from 2010 census *blocks*, which is finer — one known source of small differences).

⏱ **This is the heaviest cell — expect ~2–10 minutes for a typical state.**

> Two bugs from the old notebook fixed here:
> 1. The grid was built over the **entire continental U.S.** (~57 million cells) instead of just the state.
> 2. `sjoin` duplicated boundary cells (one copy per intersecting tract) *before* `overlay` intersected
>    each copy with every tract again — double-counting population along every tract boundary.
>    We now de-duplicate after the sjoin pre-filter.

In [ ]:
tracts_ea = tracts.to_crs(EQUAL_AREA_CRS)

# --- build the grid (vectorized) ---
minx, miny, maxx, maxy = tracts_ea.total_bounds
minx, miny = np.floor(minx / CELL_SIZE) * CELL_SIZE, np.floor(miny / CELL_SIZE) * CELL_SIZE
xs = np.arange(minx, maxx + CELL_SIZE, CELL_SIZE)
ys = np.arange(miny, maxy + CELL_SIZE, CELL_SIZE)
xx, yy = np.meshgrid(xs[:-1], ys[:-1])
xx, yy = xx.ravel(), yy.ravel()
grid = gpd.GeoDataFrame(
    {"cell_id": np.arange(xx.size)},
    geometry=shapely.box(xx, yy, xx + CELL_SIZE, yy + CELL_SIZE),
    crs=EQUAL_AREA_CRS,
)

# keep only cells touching the state — and DROP the duplicates sjoin creates
hits = gpd.sjoin(grid, tracts_ea[["geometry"]], how="inner", predicate="intersects")
grid = grid.loc[hits.index.unique()].reset_index(drop=True)
print(f"{len(grid):,} grid cells cover {STATE_NAME}")

In [ ]:
# --- intersect with tracts and allocate by area share ---
pieces = gpd.overlay(
    grid,
    tracts_ea[["GEOID", "totalpop", "tothouseholds", "novehicle", "geometry"]],
    how="intersection", keep_geom_type=True,
)
pieces["piece_area"] = pieces.geometry.area
pieces = pieces.merge(
    pieces.groupby("GEOID")["piece_area"].sum().rename("tract_area"), on="GEOID"
)
share = pieces["piece_area"] / pieces["tract_area"]
pieces["piece_pop"]          = pieces["totalpop"].fillna(0)      * share
pieces["piece_hh_novehicle"] = pieces["novehicle"].fillna(0)     * share

# sanity check: allocation must conserve totals
assert np.isclose(pieces["piece_pop"].sum(), tracts["totalpop"].sum(), rtol=1e-6)
print(f"{len(pieces):,} tract-cell pieces | population conserved: "
      f"{pieces['piece_pop'].sum():,.0f}")

---
# 7 — Supermarkets from OpenStreetMap (Overpass API)

One bbox query = the state's bounds **plus a 20-mile buffer**, so a store just across the state
line still counts for border tracts (the old query was limited to the state polygon, inflating
distances near every border).

⏱ ~30 s–2 min. The query is sent by **POST with an explicit User-Agent** (the main server rejects long GET
URLs / anonymous clients with `406 Not Acceptable`), and automatically falls back to a mirror
and retries. If every attempt fails (busy server), wait a few minutes and re-run this cell.

**Historical ("attic") data:** the query is pinned to `STORE_SNAPSHOT_DATE` with Overpass's
`[date:"…"]` setting, so it returns the supermarkets as OSM knew them *on that date* — end-2019 by
default, to match the USDA 2019 Atlas. Set `STORE_SNAPSHOT_DATE = None` in Section 1 to get
today's stores instead (e.g. to map how access has changed since 2019). Attic data exists back to
~2012. One honest caveat: OSM's *coverage* was thinner in 2019 — some stores that existed then
simply hadn't been mapped yet — so the 2019 snapshot undercounts a little more than today's data.

In [ ]:
wgs_bounds = tracts.to_crs("EPSG:4326").total_bounds  # (w, s, e, n)
buf_deg = STORE_BUFFER_MILES * M_PER_MILE / 111_320    # ~miles -> degrees latitude
w, s, e, n = (wgs_bounds[0] - buf_deg, wgs_bounds[1] - buf_deg,
              wgs_bounds[2] + buf_deg, wgs_bounds[3] + buf_deg)

shop_regex = "^(" + "|".join(SHOP_TAGS) + ")$"
# pin the query to the snapshot date (Overpass "attic data") unless None
date_setting = f'[date:"{STORE_SNAPSHOT_DATE}"]' if STORE_SNAPSHOT_DATE else ""
query = f"""[out:json][timeout:300]{date_setting};
(
  node["shop"~"{shop_regex}"]({s},{w},{n},{e});
  way["shop"~"{shop_regex}"]({s},{w},{n},{e});
  relation["shop"~"{shop_regex}"]({s},{w},{n},{e});
);
out center;"""

# POST (not GET) + a real User-Agent, with fallback mirrors. Overpass rejects long GET
# URLs / anonymous clients with errors like "406 Not Acceptable".
OVERPASS_SERVERS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
]
HEADERS = {"User-Agent": "food-desert-analysis-notebook/1.0 (research; contact via census API key holder)"}

import time
elements, last_err = None, None
for server in OVERPASS_SERVERS:
    for attempt in range(2):
        try:
            resp = requests.post(server, data={"data": query},
                                 headers=HEADERS, timeout=360)
            resp.raise_for_status()
            elements = resp.json()["elements"]
            print(f"Fetched from {server}")
            break
        except Exception as err:
            last_err = err
            print(f"{server} attempt {attempt+1} failed: {err} - retrying in 15 s")
            time.sleep(15)
    if elements is not None:
        break
if elements is None:
    raise RuntimeError(f"All Overpass servers failed; wait a few minutes and re-run. Last error: {last_err}")

rows = []
for el in elements:
    lat = el.get("lat") or el.get("center", {}).get("lat")
    lon = el.get("lon") or el.get("center", {}).get("lon")
    if lat and lon:
        rows.append({"name": el.get("tags", {}).get("name", "Unknown"),
                     "lat": lat, "lon": lon})
stores = gpd.GeoDataFrame(
    pd.DataFrame(rows),
    geometry=[Point(r["lon"], r["lat"]) for r in rows],
    crs="EPSG:4326",
).to_crs(EQUAL_AREA_CRS)
when = STORE_SNAPSHOT_DATE or "today"
print(f"{len(stores)} supermarkets found (state + {STORE_BUFFER_MILES}-mile buffer, as of {when})")
stores.head()

---
# 8 — Distance to the Nearest Supermarket

**Straight-line distance from each grid-cell centroid to the nearest store, in an equal-area
projection.** Per the FARA documentation, USDA measures grid-center-to-grid-center distance —
it is *not* a road-network distance. So switching from `osmnx`/Dijkstra to a KDTree is not just
~1000× faster and dependency-free, it's also *more faithful* to the official method.
(The road-network version systematically over-estimated distance, pushing tracts toward
"low access".) ⏱ seconds.

In [ ]:
cell_centroids = grid.geometry.centroid
tree = cKDTree(np.c_[stores.geometry.x, stores.geometry.y])
dist_m, _ = tree.query(np.c_[cell_centroids.x, cell_centroids.y])
grid["dist_miles"] = dist_m / M_PER_MILE

pieces = pieces.merge(grid[["cell_id", "dist_miles"]], on="cell_id", how="left")
print(f"Median grid distance to nearest store: {grid['dist_miles'].median():.2f} miles")

### 8.2 — Distance Heat Map (½-km grid)

Every grid cell colored by its distance to the nearest supermarket — reversed viridis, so
**yellow = closest**, dark purple = farthest. Black dots are the stores themselves.
(~a minute to draw: it's a few hundred thousand polygons.)

In [ ]:
grid_plot   = grid.to_crs("EPSG:4326")
state_line  = tracts.to_crs("EPSG:4326").dissolve().boundary
stores_plot = stores.to_crs("EPSG:4326")

fig, ax = plt.subplots(figsize=(9, 9))
grid_plot.plot(column="dist_miles", cmap="viridis_r", linewidth=0, ax=ax, legend=True,
               legend_kwds={"label": "Distance to nearest supermarket (miles)", "shrink": 0.7})
state_line.plot(ax=ax, color="black", linewidth=0.8)
stores_plot.plot(ax=ax, color="black", markersize=3)

# zoom to the state (the store layer extends into the buffer)
xmin, ymin, xmax, ymax = tracts.to_crs("EPSG:4326").total_bounds
ax.set_xlim(xmin - 0.1, xmax + 0.1); ax.set_ylim(ymin - 0.1, ymax + 0.1)

snap = STORE_SNAPSHOT_DATE[:10] if STORE_SNAPSHOT_DATE else "current"
ax.set_title(f"Distance to Nearest Supermarket — {STATE_NAME} (stores: {snap})", fontsize=11)
ax.scatter([], [], color="black", s=8, label="Supermarket")
ax.legend(loc="lower right", fontsize=8)
add_map_elements(ax)
plt.tight_layout(); plt.show()

---
# 9 — Low-Access & LILA Flags (tract level)

The distance test is applied to **every** tract (the old notebook only tested low-income tracts
against 1/10 miles and non-low-income tracts against the vehicle criteria — that hybrid doesn't
match any single USDA measure, which broke the comparison with `LA1and10`). We compute the two
official measures separately:

- **`LA_1and10`** — ≥ 500 people or ≥ 33 % beyond 1 mi (urban) / 10 mi (rural)
- **`LA_vehicle`** — ≥ 100 no-vehicle households beyond ½ mi, or ≥ 500 people / 33 % beyond 20 mi
- **`LILA_*`** = `low_income` AND the corresponding access flag

In [ ]:
pieces = pieces.merge(tracts[["GEOID", "urban"]], on="GEOID", how="left")
thr = np.where(pieces["urban"], URBAN_DIST, RURAL_DIST)

pieces["pop_far_1_10"]   = pieces["piece_pop"]          * (pieces["dist_miles"] > thr)
pieces["pop_far_20"]     = pieces["piece_pop"]          * (pieces["dist_miles"] > FAR_DIST)
pieces["hhnv_far_half"]  = pieces["piece_hh_novehicle"] * (pieces["dist_miles"] > HALF_MILE)

agg = pieces.groupby("GEOID")[["piece_pop", "pop_far_1_10", "pop_far_20", "hhnv_far_half"]].sum()
agg["share_far_1_10"] = (agg["pop_far_1_10"] / agg["piece_pop"].replace(0, np.nan) * 100).fillna(0)
agg["share_far_20"]   = (agg["pop_far_20"]   / agg["piece_pop"].replace(0, np.nan) * 100).fillna(0)

agg["LA_1and10"]  = ((agg["pop_far_1_10"] >= 500) | (agg["share_far_1_10"] >= 33)).astype(int)
agg["LA_vehicle"] = ((agg["hhnv_far_half"] >= 100) |
                     (agg["pop_far_20"] >= 500) | (agg["share_far_20"] >= 33)).astype(int)

tracts = tracts.merge(agg.reset_index()[["GEOID", "LA_1and10", "LA_vehicle",
                                         "pop_far_1_10", "share_far_1_10"]],
                      on="GEOID", how="left")
tracts["LILA_1and10"]  = (tracts["low_income"].eq(1) & tracts["LA_1and10"].eq(1)).astype(int)
tracts["LILA_vehicle"] = (tracts["low_income"].eq(1) & tracts["LA_vehicle"].eq(1)).astype(int)

print(tracts[["low_income", "LA_1and10", "LILA_1and10", "LILA_vehicle"]].sum())

---
# 10 — Maps

Three-panel classification maps (compact layout — "Latitude" labeled on the first panel only,
since the panels share the y-axis). Plotted in EPSG:4326 so the axes read as longitude/latitude.

In [ ]:
plot_gdf = tracts.to_crs("EPSG:4326")
boundary = plot_gdf.dissolve().boundary

panels = [("low_income",  "green", "Low-Income"),
          ("LA_1and10",   "blue",  "Low-Access (1 & 10 mi)"),
          ("LILA_1and10", "red",   "Low-Income & Low-Access (LILA)")]

fig, axes = plt.subplots(1, 3, figsize=(14, 6.5), sharey=True)
for k, (ax, (col, clr, title)) in enumerate(zip(axes, panels)):
    plot_gdf.plot(color=np.where(plot_gdf[col].eq(1), clr, "white"),
                  edgecolor="black", linewidth=0.2, alpha=0.6, ax=ax)
    boundary.plot(ax=ax, color="gray", linewidth=1.0)
    ax.set_title(f"{title}\n{STATE_NAME} ({YEAR})", fontsize=10)
    ax.legend(handles=[Patch(facecolor=clr, edgecolor="black", label=title),
                       Patch(facecolor="white", edgecolor="black", label="Other tracts")],
              loc="upper right", fontsize=7, title="Tract status", title_fontsize=7)
    add_map_elements(ax, ylabel=(k == 0))
fig.subplots_adjust(wspace=0.04)
plt.show()

---
# 11 — Accuracy vs. the Official USDA 2019 Atlas

The USDA file is read **directly from the ERS URL** with pandas (it's ~30 MB and the Excel
parse takes a couple of minutes — be patient, no manual download needed). We compare, tract
by tract:

| Ours | USDA column |
|---|---|
| `low_income` | `LowIncomeTracts` |
| `LA_1and10` | `LA1and10` |
| `LILA_1and10` | `LILATracts_1And10` |
| `LILA_vehicle` | `LILATracts_Vehicle` |

In [ ]:
FARA_URL = ("https://ers.usda.gov/sites/default/files/_laserfiche/DataFiles/80591/"
            "FoodAccessResearchAtlasData2019.xlsx")
usda = pd.read_excel(FARA_URL, sheet_name="Food Access Research Atlas",
                     dtype={"CensusTract": str})
usda = usda[usda["State"].str.strip() == STATE_NAME].copy()
usda["GEOID"] = usda["CensusTract"].str.zfill(11)
print(f"{len(usda)} USDA tracts for {STATE_NAME}")

In [ ]:
COMPARISONS = [("low_income",   "LowIncomeTracts",    "Low Income"),
               ("LA_1and10",    "LA1and10",           "Low Access (1&10)"),
               ("LILA_1and10",  "LILATracts_1And10",  "LILA (1&10)"),
               ("LILA_vehicle", "LILATracts_Vehicle", "LILA (vehicle)")]

usda_cols = ["GEOID"] + [u for _, u, _ in COMPARISONS] + ["Urban"]
comp = tracts.merge(usda[usda_cols], on="GEOID", how="inner")
print(f"Matched {len(comp)} / {len(tracts)} tracts\n")

for ours, theirs, label in COMPARISONS:
    match = (comp[ours] == comp[theirs]).sum()
    both  = ((comp[ours] == 1) & (comp[theirs] == 1)).sum()
    only_us   = ((comp[ours] == 1) & (comp[theirs] == 0)).sum()
    only_usda = ((comp[ours] == 0) & (comp[theirs] == 1)).sum()
    print(f"{label:<20} accuracy {match/len(comp)*100:5.1f}%  "
          f"(both: {both}, ours only: {only_us}, USDA only: {only_usda})")

urb_match = (comp["urban"].fillna(False).astype(int) == comp["Urban"]).mean() * 100
print(f"\nUrban/rural proxy agreement with USDA: {urb_match:.1f}%")

In [ ]:
# Agreement maps: white = agree, red = USDA-only (we missed), orange = ours-only (we over-flag)
comp_plot = comp.to_crs("EPSG:4326")
cmap = ListedColormap(["orange", "white", "red"])   # values -1, 0, +1

fig, axes = plt.subplots(1, 4, figsize=(17, 5.8), sharey=True)
for k, (ax, (ours, theirs, label)) in enumerate(zip(axes, COMPARISONS)):
    diff = comp_plot[theirs] - comp_plot[ours]
    comp_plot.assign(diff=diff).plot(column="diff", cmap=cmap, vmin=-1, vmax=1,
                                     edgecolor="black", linewidth=0.15, ax=ax)
    ax.set_title(f"{label}\nagreement with USDA", fontsize=10)
    ax.legend(handles=[Patch(facecolor="white",  edgecolor="black", label="Agree"),
                       Patch(facecolor="red",    edgecolor="black", label="USDA only (missed)"),
                       Patch(facecolor="orange", edgecolor="black", label="Ours only (extra)")],
              loc="upper right", fontsize=7)
    add_map_elements(ax, ylabel=(k == 0))
fig.subplots_adjust(wspace=0.04)
plt.show()

---
# 12 — Save Results

Saved as a **GeoPackage** rather than a shapefile — shapefiles truncate column names to 10
characters, which is why the old files ended up with `tothouseho`, `grid_popul`, `percentpov`…
A CSV of the flags is saved too for quick joins elsewhere.

In [ ]:
out_cols = ["GEOID", "NAMELSAD", "totalpop", "percentpov", "medfam", "urban",
            "metro_name", "low_income", "LA_1and10", "LA_vehicle",
            "LILA_1and10", "LILA_vehicle", "pop_far_1_10", "share_far_1_10", "geometry"]
tracts[out_cols].to_file(f"{STATE_NAME}_food_desert_{YEAR}.gpkg", driver="GPKG")
tracts[out_cols].drop(columns="geometry").to_csv(
    f"{STATE_NAME}_food_desert_{YEAR}.csv", index=False)
print("Saved:",
      f"{STATE_NAME}_food_desert_{YEAR}.gpkg and .csv")

---
# Methodology Notes — what changed vs. the old notebooks & why results still won't hit 100 %

### Fixes to the old calculation (these were genuine errors)
1. **Mixed-up low-access rule** — the 1/10-mile test was only applied to low-income tracts and the
   vehicle test only to non-low-income tracts. USDA applies each measure to *all* tracts; LILA is
   then the intersection with low income. Fixed in Section 9.
2. **Rural test** — `totalpop < 2500` tested the tract's own population; USDA rural means the tract
   centroid is outside any Census urban area. Fixed with the SF1 urban-share proxy (Section 3.3).
3. **State / metro thresholds** — were the *median of tract medians*; USDA (NMTC) uses the actual
   state- and CBSA-level ACS median family income, now fetched directly (Section 3.2).
4. **Grid double counting** — `sjoin` before `overlay` duplicated every boundary cell per
   intersecting tract, so overlay produced duplicate pieces and inflated population along tract
   borders; also the grid covered the whole continental US (~57 M cells). Both fixed in Section 6.
5. **Distance metric** — road-network (osmnx) distance ≠ USDA method; the Atlas uses straight-line
   grid-center distances. Fixed in Section 8 (also removes the giant road-network download).
6. **Store list** — `retail`, `farm`, `health_food`, `department_store` cast far too wide a net;
   now `supermarket` + `wholesale` only (configurable via `SHOP_TAGS`).
7. **Border effect** — stores were only queried inside the state, overstating distance for border
   tracts; now a 20-mile buffered bbox.
8. `grid_id = index` + `groupby('grid_id')` was a no-op (every index already unique).

### Remaining, *expected* sources of disagreement with USDA
- **Store directory**: USDA uses the proprietary 2019 TDLinx + SNAP STARS directories; we use
  OpenStreetMap, which misses some stores and includes others. This is the single biggest driver
  of low-access mismatches. The end-2019 attic snapshot keeps the *timing* honest, but OSM's
  coverage was thinner back then, so the 2019 snapshot undercounts somewhat more than current data.
- **Population vintage/allocation**: USDA allocates 2010 *block-level* census counts to the grid;
  we spread 2019 ACS tract totals uniformly by area.
- **Income vintage**: USDA 2019 Atlas uses 2014–18 ACS; we use 2015–19 ACS (the `YEAR=2019` API).
  Swap `ACS_BASE` to 2018 if you want an exact vintage match.
- **Urban/rural proxy**: majority-urban-population vs. centroid-in-urban-area disagree for a small
  number of fringe tracts (the agreement % is printed in Section 11).